# <span style="color:#F57C00">Turkey City District Code Table (Central Primary Key)</span>

Builds the single reference table `city_code | city_name | district_code | district_name` from the two TÜİK MEDAS exports
`il_nufus_verisi.csv` (`Adana-1`) and `ilce_nufus_verisi.csv` (`Adana(Aladağ)-1757`).
Every other district-level table can be joined on `district_code` (or `city_code` for provinces) to get consistent names and codes.
The table reflects the 2025 administrative structure; it covers every district that exists from 2013 onward.
Input and output live in the same folder (`data/raw/`); `district_name` is **not** unique (e.g. 51 × "Merkez"), only `district_code` is.

In [ ]:
# Central reference table: city_code | city_name | district_code | district_name  (join key for every district-level table)
import os
import pandas as pd

DATA_DIR = "../data/raw"  # inputs and output share this folder
CITY_PATH = os.path.join(DATA_DIR, "il_nufus_verisi.csv")
DISTRICT_PATH = os.path.join(DATA_DIR, "ilce_nufus_verisi.csv")
CODE_TABLE_PATH = os.path.join(DATA_DIR, "turkey_city_district_code_table.csv")

# 'Adana-1'  /  'Adana(Aladağ)-1757'  ->  name(s) + code
CITY_LABEL_PATTERN = r'^(?P<city_name>.*)-(?P<city_code>\d+)$'
DISTRICT_LABEL_PATTERN = r'^(?P<city_name>.*?)\((?P<district_name>.*)\)-(?P<district_code>\d+)$'


def _parse_labels(path: str, pattern: str) -> pd.DataFrame:
    """MEDAS pivot export -> one row per 'Name-Code' label; header/separator rows are NaN and dropped."""
    labels = pd.read_csv(
        path, sep='|', header=None, usecols=[1], names=['label'], encoding='utf-8-sig',
    )['label'].dropna().str.strip()
    parsed = labels.str.extract(pattern)
    if parsed.isna().any(axis=None):
        raise ValueError(f"Unparseable labels in {path}: {labels[parsed.isna().any(axis=1)].tolist()}")
    return parsed


def build_district_code_table(city_path: str, district_path: str) -> pd.DataFrame:
    cities = _parse_labels(city_path, CITY_LABEL_PATTERN)
    districts = _parse_labels(district_path, DISTRICT_LABEL_PATTERN)

    # The district labels carry the city *name*; the city code comes from the city file.
    table = districts.merge(cities, on='city_name', how='left', validate='many_to_one')
    if table['city_code'].isna().any():
        raise ValueError(f"Cities missing from {city_path}: {sorted(table.loc[table['city_code'].isna(), 'city_name'].unique())}")
    if table['district_code'].duplicated().any():
        raise ValueError(f"Duplicate district_code: {sorted(table.loc[table['district_code'].duplicated(), 'district_code'].unique())}")

    return (
        table.astype({'city_code': int, 'district_code': int})[['city_code', 'city_name', 'district_code', 'district_name']]
        .sort_values(['city_code', 'district_code'], ignore_index=True)
    )


df_codes = build_district_code_table(CITY_PATH, DISTRICT_PATH)
df_codes.to_csv(CODE_TABLE_PATH, index=False)
print(f"{len(df_codes)} districts across {df_codes['city_code'].nunique()} cities -> {CODE_TABLE_PATH}")
df_codes.head(5)


# <span style="color:#F57C00">District-Level Preprocessing Pipeline (GeoPandas & TÜİK ID Integration)</span>

This notebook establishes the spatial data preprocessing pipeline for the **AI-Driven Urban Green Planner**.
It uses **GeoPandas** to:
1. Ingest raw UN OCHA administrative boundaries from HDX (districts and cities), keeping only the columns we need.
2. Join them to the central code table built above: `city_code` (secondary key) from the OCHA P-code, `district_code` (primary key) from a name match inside each city.
3. Validate the joins (strict 1:1) and the spatial centroids (100% Point-in-Polygon check).
4. Export the single sources of truth: `turkey_districts_master.geojson` and `turkey_cities_master.geojson`.
5. Render spatial verification boundary maps.

## 1. Imports and Paths Configuration
We use `geopandas`, `pandas`, `shapely`, and `matplotlib` within the `general_use` conda environment.

In [ ]:
import os
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

RAW_DISTRICTS_PATH = "../data/raw/turkey_districts_raw.geojson"
RAW_CITIES_PATH = "../data/raw/turkey_provinces_raw.geojson"
CODE_TABLE_PATH = "../data/raw/turkey_city_district_code_table.csv"
PROCESSED_GEOJSON_PATH = "../data/processed/turkey_districts_master.geojson"
PROCESSED_CITIES_GEOJSON_PATH = "../data/processed/turkey_cities_master.geojson"

os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../data/results", exist_ok=True)
print("GeoPandas environment configured.")


## 2. Ingest Boundaries & Join the Central Code Table
Load the raw district and city boundaries (`columns=` drops the constant/empty OCHA attributes at read time) and join both to the central code table:
- **Districts**, on **both** keys in a single merge:
  - **City:** `city_code` is the plate code inside the OCHA `adm1_pcode` (`TUR034` → `34`).
  - **District:** a normalized `district_name` (accents, case, punctuation) matched **within the city**, so the 51 × "Merkez" never collide. OCHA names the central district of non-metropolitan cities after the city (`Adiyaman`), so the code table's `Merkez` rows are keyed by their city name.
- **Cities:** `city_code` from `adm1_pcode`, names from the code table; the OCHA city name is only a sanity check of the plate-code extraction.

Both merges are strictly 1:1 in both directions and raise if any polygon or code is left over (this also catches a wrong city code).

In [ ]:
# `columns=` drops everything we do not need (constant/empty OCHA attributes: adm0_*, lang*, version, valid_*, ...) at read time
DISTRICT_COLUMNS = ["adm2_name1", "adm1_pcode", "area_sqkm", "center_lat", "center_lon"]
CITY_COLUMNS = ["adm1_pcode", "adm1_name", "area_sqkm", "center_lat", "center_lon"]

gdf_raw = gpd.read_file(RAW_DISTRICTS_PATH, columns=DISTRICT_COLUMNS)
gdf_cities_raw = gpd.read_file(RAW_CITIES_PATH, columns=CITY_COLUMNS)
df_codes = pd.read_csv(CODE_TABLE_PATH)

print(f"Raw district boundaries loaded: {len(gdf_raw)} features.")
print(f"Raw city boundaries loaded: {len(gdf_cities_raw)} features.")
print(f"Central code table loaded: {len(df_codes)} districts in {df_codes['city_code'].nunique()} cities.")


In [ ]:
def name_key(names: pd.Series) -> pd.Series:
    """Accent/case/punctuation-insensitive match key: 'Çukurova' and 'Cukurova' -> 'cukurova', 'Iğdır' -> 'igdir'."""
    return names.str.normalize('NFKD').str.lower().str.replace('ı', 'i').str.replace(r'[^a-z0-9]', '', regex=True)


# OCHA side: the city plate code sits inside the P-code ('TUR034' -> 34)
ocha = gdf_raw.assign(city_code=gdf_raw['adm1_pcode'].str[-3:].astype(int), key=name_key(gdf_raw['adm2_name1']))
# Code table side: TÜİK calls the central district of a non-metropolitan city 'Merkez', OCHA names it after the city
codes = df_codes.assign(key=name_key(df_codes['district_name'].where(df_codes['district_name'] != 'Merkez', df_codes['city_name'])))

# Strict 1:1 in both directions (this also catches a wrong city_code): raises if any polygon or code is left over
merged = ocha.merge(codes, on=['city_code', 'key'], how='outer', validate='one_to_one', indicator=True)
if (merged['_merge'] != 'both').any():
    raise ValueError(merged.loc[merged['_merge'] != 'both', ['_merge', 'city_code', 'adm2_name1', 'district_code', 'district_name']].to_string(index=False))

gdf = (
    merged[['district_code', 'city_code', 'district_name', 'city_name', 'area_sqkm', 'center_lat', 'center_lon', 'geometry']]
    .round({'area_sqkm': 2, 'center_lat': 6, 'center_lon': 6})
    .sort_values(['city_code', 'district_code'], ignore_index=True)
)

# Point-in-polygon verification (vectorized; 'intersects' = inside or on the boundary)
centroids = gpd.GeoSeries(gpd.points_from_xy(gdf['center_lon'], gdf['center_lat']), index=gdf.index, crs=gdf.crs)
print(f"Spatial Validation: {gdf.geometry.intersects(centroids).sum()} / {len(gdf)} centroids inside district boundaries.")
print(f"Code Validation: {len(gdf)} / {len(gdf_raw)} districts joined to the central code table (strict 1:1).")


In [ ]:
# Cities: same plate-code extraction, names from the code table. The OCHA name is only a sanity check of the extraction.
gdf_cities = gdf_cities_raw.assign(city_code=gdf_cities_raw['adm1_pcode'].str[-3:].astype(int)).merge(
    df_codes[['city_code', 'city_name']].drop_duplicates(),
    on='city_code', how='outer', validate='one_to_one', indicator=True,
)
bad = (gdf_cities['_merge'] != 'both') | (name_key(gdf_cities['adm1_name']) != name_key(gdf_cities['city_name']))
if bad.any():
    raise ValueError(gdf_cities.loc[bad, ['_merge', 'city_code', 'adm1_pcode', 'adm1_name', 'city_name']].to_string(index=False))

gdf_cities = (
    gdf_cities[['city_code', 'city_name', 'area_sqkm', 'center_lat', 'center_lon', 'geometry']]
    .round({'area_sqkm': 2, 'center_lat': 6, 'center_lon': 6})
    .sort_values('city_code', ignore_index=True)
)

centers = gpd.GeoSeries(gpd.points_from_xy(gdf_cities['center_lon'], gdf_cities['center_lat']), index=gdf_cities.index, crs=gdf_cities.crs)
area_gap = (gdf.groupby('city_code')['area_sqkm'].sum() - gdf_cities.set_index('city_code')['area_sqkm']).abs().max()
print(f"City Validation: {len(gdf_cities)} cities joined, {gdf_cities.geometry.intersects(centers).sum()} / {len(gdf_cities)} centers inside their polygon, "
      f"max |sum of district areas - city area| = {area_gap:.2f} sq km.")


## 3. Export Master GeoJSON Datasets
Save `turkey_districts_master.geojson` (973 district polygons, centroids, the central keys `district_code` primary / `city_code` secondary, official TÜİK names) and `turkey_cities_master.geojson` (81 city polygons, centers, `city_code`, official TÜİK names).

In [ ]:
gdf.to_file(PROCESSED_GEOJSON_PATH, driver="GeoJSON")
gdf_cities.to_file(PROCESSED_CITIES_GEOJSON_PATH, driver="GeoJSON")
print(f"Saved Master GeoJSON to {PROCESSED_GEOJSON_PATH} ({os.path.getsize(PROCESSED_GEOJSON_PATH) / (1024*1024):.2f} MB)")
print(f"Saved Cities GeoJSON to {PROCESSED_CITIES_GEOJSON_PATH} ({os.path.getsize(PROCESSED_CITIES_GEOJSON_PATH) / (1024*1024):.2f} MB)")
print("Sample Master Records:")
print(gdf.drop(columns="geometry").head(5).to_string(index=False))
print("Sample City Records:")
print(gdf_cities.drop(columns="geometry").head(3).to_string(index=False))


## 4. GeoPandas Direct Map Visualization
Render all 973 district boundaries and center coordinates using GeoPandas built-in plotting.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 7))
gdf.plot(ax=ax, color="#e5f5e0", edgecolor="#31a354", linewidth=0.4)
ax.scatter(gdf["center_lon"], gdf["center_lat"], color="#006d2c", s=1.5, alpha=0.7, label="District Centroids")
ax.set_title("Türkiye 973 District Administrative Boundaries & Centroids (with TÜİK IDs)", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Longitude (°E)")
ax.set_ylabel("Latitude (°N)")
ax.legend(loc="lower right")
plt.tight_layout()
overview_path = "../data/results/turkey_districts_overview.png"
plt.savefig(overview_path, dpi=200, bbox_inches="tight")
print(f"District boundary plot saved to {overview_path}")
plt.show()

# <span style="color:#F57C00">TÜİK MEDAS District Population: Clean, Combine and Match to the Code Table</span>

The 18 files in `data/raw/nufus_ilce/` were downloaded by hand from [TÜİK MEDAS](https://biruni.tuik.gov.tr/medas/?locale=tr) and numbered `1.csv` ... `18.csv`.
Every file holds **all years (2007-2025)** for a group of districts, so the 18 files together cover all districts.
Each row is one district in one year, with the population split by gender and 5-year age group (38 columns).

What we do, step by step:
1. Read the 18 files and put them one under the other (`pd.concat`).
2. Clean the rows: fill in the year, drop the empty rows, take the district code out of the `Giresun(Alucra)-1133` label.
3. Give the 38 age columns short English names (`male_0_4`, `female_90plus`, ...) and add the totals.
4. Keep the years from 2013 and match every row to the central code table, which adds `city_code`, `city_name` and `district_name`.
5. Save the result to `data/processed/tuik_districts_population_clean.csv`.

**Read this before using the numbers.** Every column name ends with `ve Şehir`: the files contain the population of the *urban* settlements (`Şehir`) of each district, **not** the whole district population. The villages (`Köy`) of the non-metropolitan provinces are not included. Two signs of this:
- the 81 city totals of 2025 add up to 80.6 million, below Turkey's population (about 86 million);
- Adana/Aladağ jumps from 4,224 (2012) to 17,221 (2013). This fits the 2012 metropolitan law (no. 6360), after which the villages of the 30 metropolitan provinces became neighborhoods.

That jump is also why we start in **2013**: 350 districts change by more than 40% between 2012 and 2013, but only a few districts per year after that.

## Step 1: Read and combine the 18 files
Each file is separated by `|` and starts with two title lines, so we skip them. We read the files one by one into a list and glue the list together with `pd.concat`.

In [ ]:
from pathlib import Path
import pandas as pd

POP_DIR = Path("../data/raw/nufus_ilce")
POP_OUTPUT_CSV = "../data/processed/tuik_districts_population_clean.csv"
START_YEAR = 2013  # first year with a consistent definition (see the note above)

# 1.csv, 2.csv, ... 18.csv. Sorting by the number (not by the text) puts 2.csv before 10.csv.
files = sorted(POP_DIR.glob("*.csv"), key=lambda path: int(path.stem))

frames = []
for file in files:
    # skiprows=2: the first two lines are titles, the third line holds the column names
    frames.append(pd.read_csv(file, sep="|", skiprows=2))

raw = pd.concat(frames, ignore_index=True)
print(f"{len(files)} files -> {raw.shape[0]} rows and {raw.shape[1]} columns")
raw.iloc[:6, :4]  # the first rows as they come out of the files: messy, with empty cells


## Step 2: Clean the rows
- The year is written only on the first row of each block, so we copy it downwards (`ffill`).
- Rows without a district label are empty separator rows, we drop them.
- The label `Giresun(Alucra)-1133` ends with the TÜİK district code: we split at `-` and take the last part.
- An empty cell in an age column means nobody lives in that group, so we fill it with 0.

In [ ]:
# The age columns have ' ve ' in their name, e.g. 'Erkek ve 0-4 ve Şehir'.
# We keep them together with the year and the label; everything else (like the empty last column) is dropped.
raw = raw.rename(columns={"Unnamed: 0": "year", "Unnamed: 1": "label"})
age_columns = [column for column in raw.columns if " ve " in column]

df = raw[["year", "label"] + age_columns].copy()
df["year"] = df["year"].ffill()
df["district_code"] = df["label"].str.split("-").str[-1]  # empty separator rows stay empty here
df = df.dropna(subset=["label"])
df = df.astype({"year": int, "district_code": int})

# 99% of the empty cells are in the oldest groups (85+), where small districts have nobody
df[age_columns] = df[age_columns].fillna(0).astype(int)

# The same district and year must not appear twice (that would mean two downloads overlap)
if df.duplicated(subset=["year", "district_code"]).any():
    raise ValueError("The same district and year appear in more than one file")

print(f"{len(df)} rows | years {df['year'].min()}-{df['year'].max()} | {df['district_code'].nunique()} different districts")
df[["year", "label", "district_code"]].head(3)


## Step 3: Short English column names and totals
A MEDAS column name glues three things together: gender, age group and the settlement type, e.g. `Erkek ve 0-4 ve Şehir`. The last part (`Şehir`) is the same in every column, so it is useless and we drop it. What is left becomes one short English name: `male_0_4`, `female_90plus`, ...
Then we add the totals with `sum(axis=1)` (a sum across the columns of every row): `male_pop`, `female_pop`, `total_pop` for all ages, and `total_0_4`, `total_5_9`, ... for men + women in one age group.

In [ ]:
AGE_GROUPS = [
    "0_4", "5_9", "10_14", "15_19", "20_24", "25_29", "30_34", "35_39", "40_44", "45_49",
    "50_54", "55_59", "60_64", "65_69", "70_74", "75_79", "80_84", "85_89", "90plus",
]
GENDERS = {"Erkek": "male", "Kadın": "female"}
assert len(age_columns) == 2 * len(AGE_GROUPS), "unexpected number of age columns"

new_names = {}
for column in age_columns:
    gender_tr, age, _ = column.split(" ve ")  # 'Erkek ve 0-4 ve Şehir' -> 'Erkek', '0-4', 'Şehir'
    age = age.replace("-", "_").replace("+", "plus")  # '0-4' -> '0_4' and '90+' -> '90plus'
    new_names[column] = f"{GENDERS[gender_tr]}_{age}"  # 'male_0_4'
df = df.rename(columns=new_names)

male_columns = [f"male_{age}" for age in AGE_GROUPS]
female_columns = [f"female_{age}" for age in AGE_GROUPS]

df["male_pop"] = df[male_columns].sum(axis=1)
df["female_pop"] = df[female_columns].sum(axis=1)
df["total_pop"] = df["male_pop"] + df["female_pop"]

for age in AGE_GROUPS:  # men + women for every age group
    df[f"total_{age}"] = df[f"male_{age}"] + df[f"female_{age}"]

df[["district_code", "year", "male_0_4", "female_0_4", "total_0_4", "total_pop"]].head(3)


## Step 4: Keep 2013+ and match to the code table
We join the population rows to the central code table on `district_code`. Many rows (one per year) belong to one district, but every code appears once in the table (`many_to_one`). This adds `city_code`, `city_name` and `district_name`, so the names are the same in every year even if a district was renamed.
If a district code is not in the table, we stop with an error instead of silently losing rows.

In [ ]:
CODE_TABLE_PATH = "../data/raw/turkey_city_district_code_table.csv"
codes = pd.read_csv(CODE_TABLE_PATH)

df = df[df["year"] >= START_YEAR]
df = df.merge(codes, on="district_code", how="left", validate="many_to_one")

not_in_table = df.loc[df["city_code"].isna(), "district_code"].unique()
if len(not_in_table) > 0:
    raise ValueError(f"These district codes are not in the code table: {not_in_table}")

print(f"{len(df)} rows from {START_YEAR} | {df['district_code'].nunique()} districts in {df['city_code'].nunique()} cities, all matched")


## Step 5: Order the columns and save
Keys first (`year`, `district_code`, `city_code` and the two names), then the totals for all ages, then `total_<age>` (men + women), and last `male_<age>` and `female_<age>`. The rows are sorted by district and year.

In [ ]:
key_columns = ["year", "district_code", "city_code", "district_name", "city_name"]
total_columns = ["total_pop", "male_pop", "female_pop"]
age_total_columns = [f"total_{age}" for age in AGE_GROUPS]

df_population = df[key_columns + total_columns + age_total_columns + male_columns + female_columns]
df_population = df_population.sort_values(["district_code", "year"], ignore_index=True)

Path(POP_OUTPUT_CSV).parent.mkdir(parents=True, exist_ok=True)
df_population.to_csv(POP_OUTPUT_CSV, index=False)

print(f"Saved {len(df_population)} rows x {df_population.shape[1]} columns to {POP_OUTPUT_CSV} "
      f"({Path(POP_OUTPUT_CSV).stat().st_size / (1024 * 1024):.2f} MB)")
df_population[key_columns + total_columns + ["total_0_4", "total_65_69"]].head(5)


# <span style="color:#F57C00">Open-Meteo Open Data (AWS): Monthly Climate for Every District</span>

For every district we take the center point (`center_lat`, `center_lon` of the master GeoJSON) and read its **hourly** weather from the free [Open-Meteo open data on AWS S3](https://github.com/open-meteo/open-data). Then we turn the hours into Turkish **days** and the days into a **monthly** table: 973 districts x 312 months (2000-2025).

**Why not the Open-Meteo API?** The free API counts a long request as many calls (`days / 14 x variables / 10`) and allows 10,000 calls per day. All districts would need about a week. The open data on S3 has no such limit and we only read the small pieces we need.

**What we read**
| | Temperature | Precipitation |
| :--- | :--- | :--- |
| Dataset | ERA5-Land (ECMWF reanalysis) | ERA5 (ECMWF reanalysis) |
| Grid | 0.1° (about 11 km) | 0.25° (about 25 km). ERA5-Land has no rain in the open data |
| Files | one file per year until 2021, then 21-day files | the same |

**How it is fast.** One connection to the bucket (in the US) is slow, about 40 KB/s. So we open **many connections at once** (`THREADS`) and every connection reads a few latitude rows (`ROWS_PER_TASK`) of the Turkey box. The finished pieces are saved in `data/raw/climate_s3/`, so you can stop at any time and run the cell again. The whole download takes about 30-40 minutes and uses 1.5 GB. `DOWNLOAD = False` by default, so "Run All" never starts it by accident.

**Good to know**
- The values are the **grid-cell values**. The API additionally corrects the temperature to the height of the point, so in mountains the API is warmer by about 1 °C (Feke: 1.2 °C). On flat land both are the same (Adana: 0.04 °C).
- ERA5-Land has no data over the sea. The center of 14 coastal districts (Adalar, Kartal, Datça, ...) lies in a sea cell: they get the **nearest land cell** (at most 0.15° away). The API does the same.
- The days follow the Turkish clock (`Europe/Istanbul`), also in the years with summer time.
- Needs two extra packages: `pip install omfiles s3fs`.

## Step 1: Settings
Change the years, the thresholds and the switches here. Everything else adapts.

In [ ]:
import hashlib
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import s3fs
from omfiles import OmFileReader

FIRST_YEAR, LAST_YEAR = 2000, 2025
HOT_DAY_C = 30  # "hot day": the daily maximum is at least 30 C
VERY_HOT_DAY_C = 35  # "very hot day": the daily maximum is at least 35 C
TROPICAL_NIGHT_C = 20  # "tropical night": the daily minimum is at least 20 C

DOWNLOAD = False  # True = read the missing pieces from AWS (about 30-40 minutes, 1.5 GB in the cache folder)
THREADS = 128  # parallel connections: one is slow (about 40 KB/s), many together are fast
BLOCK = 256 * 1024  # bytes per request
ROWS_PER_TASK = 4  # latitude rows that one thread reads with one opened file (the file header is read only once)

SOURCES = {  # short name -> where the hourly data lives in the open bucket, and the grid step in degrees
    "temp": {"dataset": "copernicus_era5_land", "variable": "temperature_2m", "step": 0.1},
    "precip": {"dataset": "copernicus_era5", "variable": "precipitation", "step": 0.25},
}
LAST_YEARLY_FILE = 2021  # until 2021 there is one file per year; after that the files hold 21 days
CHUNK_HOURS = 21 * 24

DISTRICTS_PATH = "../data/processed/turkey_districts_master.geojson"
CACHE_DIR = Path("../data/raw/climate_s3")
OUTPUT_DIR = Path("../data/processed")


## Step 2: Which grid cell belongs to which district
The grids start at -90° latitude and -180° longitude, so the cell of a point is its distance from there divided by the grid step. Many districts could share a cell, so we keep a list of the **distinct cells** and, for every district, the number of its cell. The cells are grouped by latitude row, because we read one row at a time.

In [ ]:
districts = gpd.read_file(DISTRICTS_PATH, ignore_geometry=True)  # the table only, no polygons
districts = districts[["district_code", "city_code", "district_name", "city_name", "center_lat", "center_lon"]]

for name, source in SOURCES.items():
    districts[f"{name}_y"] = ((districts["center_lat"] + 90) / source["step"]).round().astype(int)
    districts[f"{name}_x"] = ((districts["center_lon"] + 180) / source["step"]).round().astype(int)


def make_grid(name):
    points = districts[[f"{name}_y", f"{name}_x"]].set_axis(["y", "x"], axis=1)
    cells = points.drop_duplicates(ignore_index=True)  # one row per distinct cell
    cell_of_district = points.merge(cells.reset_index(names="cell"), on=["y", "x"], how="left")["cell"].to_numpy()
    rows = {}  # latitude row -> (first column, last column, numbers of the cells on this row, their columns)
    for y, group in cells.groupby("y"):
        x_first = group["x"].min()
        rows[y] = (x_first, group["x"].max(), group.index.to_numpy(), (group["x"] - x_first).to_numpy())
    signature = hashlib.md5(cells.to_numpy().tobytes()).hexdigest()[:8]  # changes when the cells change
    return {"n_cells": len(cells), "cell_of_district": cell_of_district, "rows": rows, "signature": signature}


SEA_FIX_PATH = CACHE_DIR / "sea_fix.csv"


def apply_sea_fix():
    # sea districts get the nearest land cell; the list is found and saved in Step 4
    if SEA_FIX_PATH.exists():
        fix = pd.read_csv(SEA_FIX_PATH).set_index("district_code")
        sea = districts["district_code"].isin(fix.index)
        for column in ("temp_y", "temp_x"):
            districts.loc[sea, column] = fix.loc[districts.loc[sea, "district_code"], column].to_numpy()
        grid["temp"] = make_grid("temp")


grid = {name: make_grid(name) for name in SOURCES}
apply_sea_fix()
for name in SOURCES:
    print(f"{name}: {len(districts)} districts on {grid[name]['n_cells']} distinct grid cells in {len(grid[name]['rows'])} latitude rows")


## Step 3: Which files and which hours do we need?
The open data counts time in **hours since 1970-01-01 (UTC)**. We need all hours from 24 hours before 1 January of the first year (the Turkish day starts 2-3 hours before UTC midnight) to the last hour of the last year. Until 2021 these hours are in one file per year, afterwards in files of 21 days. `pieces` is the list of reads: which file, from which hour in the file, how many hours, and where they go in our table.

In [ ]:
def hour_number(date):
    return int(pd.Timestamp(date, tz="UTC").timestamp() // 3600)


start_hour = hour_number(f"{FIRST_YEAR}-01-01") - 24
end_hour = hour_number(f"{LAST_YEAR + 1}-01-01")
total_hours = end_hour - start_hour

pieces = []
hour = start_hour
while hour < end_hour:
    if hour < hour_number(f"{LAST_YEARLY_FILE + 1}-01-01"):  # yearly files
        year = pd.Timestamp(hour * 3600, unit="s").year
        stop = min(end_hour, hour_number(f"{year + 1}-01-01"))
        pieces.append({"file": f"year_{year}.om", "begin": hour - hour_number(f"{year}-01-01"), "count": stop - hour, "offset": hour - start_hour})
    else:  # files of 21 days
        number = hour // CHUNK_HOURS
        stop = min(end_hour, (number + 1) * CHUNK_HOURS)
        pieces.append({"file": f"chunk_{number}.om", "begin": hour - number * CHUNK_HOURS, "count": stop - hour, "offset": hour - start_hour})
    hour = stop

assert sum(piece["count"] for piece in pieces) == total_hours  # no hour is missing or read twice
print(f"{total_hours:,} hours = {len(pieces)} pieces per variable ({sum(p['file'].startswith('year') for p in pieces)} yearly files, {sum(p['file'].startswith('chunk') for p in pieces)} files of 21 days)")
pd.DataFrame(pieces).iloc[[0, 1, 2, -3, -2, -1]]


## Step 4: Read the pieces (parallel, can be stopped and started again)
`RangeCache` makes the reading efficient: the file format asks for many tiny ranges, we answer them from blocks of 256 KB. Every task opens the file once and reads `ROWS_PER_TASK` latitude rows, so the file header and index blocks are loaded only once for all of them. A finished piece is saved as a `.npy` file (its name holds a short code of the grid cells, so old pieces are never used with other cells), so the next run only reads what is still missing.
First (only the very first time) we look for the **sea cells**: we read one hour of the temperature data and look for cells without data.

In [ ]:
file_sizes = {}  # size of every file, asked from S3 only once for all readers


class RangeCache:
    # Wraps s3fs for omfiles: reads aligned blocks of `block` bytes and remembers the last few of them
    def __init__(self, fs, block, keep=8):
        self.fs, self.block, self.keep, self.cache = fs, block, keep, {}

    def size(self, path):
        if path not in file_sizes:
            file_sizes[path] = self.fs.size(path)
        return file_sizes[path]

    def cat_file(self, path, start=None, end=None, **kwargs):
        start = 0 if start is None else start
        end = self.size(path) if end is None else end
        first, last = start // self.block, (end - 1) // self.block
        for b in range(first, last + 1):
            if (path, b) not in self.cache:
                if len(self.cache) >= self.keep:
                    self.cache.pop(next(iter(self.cache)))
                lo = b * self.block
                self.cache[(path, b)] = self.fs.cat_file(path, start=lo, end=min(lo + self.block, self.size(path)))
        data = b"".join(self.cache[(path, b)] for b in range(first, last + 1))
        return data[start - first * self.block:][: end - start]


fs = s3fs.S3FileSystem(anon=True, config_kwargs={"max_pool_connections": THREADS + 20})


def s3_path(name, file):
    return f"openmeteo/data/{SOURCES[name]['dataset']}/{SOURCES[name]['variable']}/{file}"


def open_reader(path):
    # every task opens its own reader (a reader and its small cache are used by one thread only)
    return OmFileReader.from_fsspec(RangeCache(fs, BLOCK), path)


def row_groups(rows):
    # [61, 62, 63, 64, 65, ...] -> [[61, 62, 63, 64], [65, ...], ...]: one task reads ROWS_PER_TASK rows
    return [rows[i:i + ROWS_PER_TASK] for i in range(0, len(rows), ROWS_PER_TASK)]


def find_sea_fix():
    # one hour of the temperature data over the whole Turkey box: a cell without data is sea
    y_first, y_last = districts["temp_y"].min(), districts["temp_y"].max()
    x_first, x_last = districts["temp_x"].min(), districts["temp_x"].max()
    path = s3_path("temp", "chunk_946.om")

    def first_hours(ys):
        reader = open_reader(path)
        return [reader[y, x_first:x_last + 1, 0:1][:, 0] for y in ys]

    with ThreadPoolExecutor(THREADS) as pool:
        parts = pool.map(first_hours, row_groups(list(range(y_first, y_last + 1))))
    box = np.stack([row for part in parts for row in part])
    land_y, land_x = np.nonzero(~np.isnan(box))
    land_y, land_x = land_y + y_first, land_x + x_first
    step = SOURCES["temp"]["step"]
    fixes = []
    for row in districts.itertuples():
        if np.isnan(box[row.temp_y - y_first, row.temp_x - x_first]):
            # distance in degrees (a degree of longitude is shorter in the north)
            distance = np.hypot((land_y - row.temp_y) * step, (land_x - row.temp_x) * step * np.cos(np.radians(row.center_lat)))
            nearest = distance.argmin()
            fixes.append({"district_code": row.district_code, "temp_y": land_y[nearest], "temp_x": land_x[nearest]})
    return pd.DataFrame(fixes)


if DOWNLOAD and not SEA_FIX_PATH.exists():
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    find_sea_fix().to_csv(SEA_FIX_PATH, index=False)
    apply_sea_fix()
    print(f"{len(pd.read_csv(SEA_FIX_PATH))} districts have their center in the sea and use the nearest land cell")

# the table of all hours: one row per grid cell, one column per hour (empty until it is read)
hourly = {name: np.full((grid[name]["n_cells"], total_hours), np.nan, dtype=np.float32) for name in SOURCES}


def cache_path(name, piece):  # the code of the grid cells is in the name: other cells never reuse old pieces
    return CACHE_DIR / f"{name}_{grid[name]['signature']}_{piece['file'][:-3]}_{piece['begin']}_{piece['count']}.npy"


tasks, remaining = [], {}  # tasks: a few latitude rows each; remaining: tasks still open per piece
for name in SOURCES:
    for number, piece in enumerate(pieces):
        path = cache_path(name, piece)
        if path.exists() and np.load(path, mmap_mode="r").shape == (grid[name]["n_cells"], piece["count"]):
            hourly[name][:, piece["offset"]:piece["offset"] + piece["count"]] = np.load(path)  # already in the cache
        else:
            groups = row_groups(list(grid[name]["rows"]))
            remaining[(name, number)] = len(groups)
            tasks += [(name, number, ys) for ys in groups]
print(f"{2 * len(pieces) - len(remaining)} of {2 * len(pieces)} pieces are in the cache, {len(remaining)} still to read ({len(tasks)} tasks)")


def read_piece_rows(name, number, ys):
    piece = pieces[number]
    reader = open_reader(s3_path(name, piece["file"]))  # opened once for all rows of this task
    for y in ys:
        x_first, x_last, cell_ids, columns = grid[name]["rows"][y]
        data = reader[y, x_first:x_last + 1, piece["begin"]:piece["begin"] + piece["count"]]  # this row, all hours of the piece
        hourly[name][cell_ids, piece["offset"]:piece["offset"] + piece["count"]] = data[columns, :]  # keep only the district cells


if not DOWNLOAD:
    print("DOWNLOAD is False, so nothing is read from AWS. Set DOWNLOAD = True in Step 1 to start.")
elif tasks:
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    started, done = time.time(), 0
    pool = ThreadPoolExecutor(THREADS)
    try:
        futures = {pool.submit(read_piece_rows, *task): task[:2] for task in tasks}
        for future in as_completed(futures):
            future.result()  # an error inside a thread is raised here
            key = futures[future]
            remaining[key] -= 1
            if remaining[key] == 0:  # all rows of this piece are in: save it
                name, number = key
                piece = pieces[number]
                np.save(cache_path(name, piece), hourly[name][:, piece["offset"]:piece["offset"] + piece["count"]])
                done += 1
                if done % 10 == 0 or done == len(remaining):
                    print(f"[{done}/{len(remaining)}] pieces saved, last: {name} {piece['file']} ({(time.time() - started) / 60:.1f} min)")
    except BaseException:
        pool.shutdown(wait=False, cancel_futures=True)  # stop the waiting reads; the saved pieces stay
        raise
    pool.shutdown()


## Step 5: From hours to days to months
Only when every hour is there. We put the hours in a table (one column per grid cell) with the **Turkish** clock as index, and let `resample` do the grouping:
- `resample("D")` makes Turkish days: the mean, the hottest hour and the coldest hour of every day (the daily rain is the sum),
- `resample("MS")` makes months from the days: the mean of the daily means, the hottest and coldest day, the **number of** hot days, very hot days and tropical nights, and the total rain.

At the end every district takes the columns of its own cell, and we add the keys (`city_code`, names) with one merge.

In [ ]:
complete = all(not np.isnan(hourly[name]).any() for name in SOURCES)

if not complete:
    print("Not finished yet: the monthly table is written when every hour is read (set DOWNLOAD = True in Step 1).")
else:
    local_hours = pd.date_range(pd.Timestamp(start_hour * 3600, unit="s", tz="UTC"), periods=total_hours, freq="h").tz_convert("Europe/Istanbul")
    wanted_days = slice(f"{FIRST_YEAR}-01-01", f"{LAST_YEAR}-12-31")  # drops the helper hours before and after

    temp = pd.DataFrame(hourly["temp"].T, index=local_hours).resample("D")  # one column per grid cell
    daily_mean, daily_max, daily_min = temp.mean().loc[wanted_days], temp.max().loc[wanted_days], temp.min().loc[wanted_days]
    # Open-Meteo stores the rain of the hour BEFORE each time stamp, so the value at 00:00 is the last hour of the previous day.
    # We keep the time stamps as they are: that one hour per day (and per month) lands one day (one month) later.
    daily_rain = pd.DataFrame(hourly["precip"].T, index=local_hours).resample("D").sum().loc[wanted_days]

    months = {  # table of months x grid cells, and the grid it belongs to
        "temp_mean": (daily_mean.resample("MS").mean(), "temp"),
        "temp_max": (daily_max.resample("MS").max(), "temp"),
        "temp_min": (daily_min.resample("MS").min(), "temp"),
        "hot_days": ((daily_max >= HOT_DAY_C).resample("MS").sum(), "temp"),
        "very_hot_days": ((daily_max >= VERY_HOT_DAY_C).resample("MS").sum(), "temp"),
        "tropical_nights": ((daily_min >= TROPICAL_NIGHT_C).resample("MS").sum(), "temp"),
        "precip_sum": (daily_rain.resample("MS").sum(), "precip"),
    }

    def per_district(table, name):
        wide = table[grid[name]["cell_of_district"]]  # every district takes the column of its own cell
        wide.columns = districts["district_code"].to_numpy()
        return wide.stack()  # one row per month and district

    monthly = pd.concat({metric: per_district(table, name) for metric, (table, name) in months.items()}, axis=1)
    monthly = monthly.rename_axis(["month_start", "district_code"]).reset_index()
    monthly["year"], monthly["month"] = monthly["month_start"].dt.year, monthly["month_start"].dt.month

    keys = districts[["district_code", "city_code", "district_name", "city_name"]]
    monthly = monthly.merge(keys, on="district_code", how="left", validate="many_to_one")
    order = ["year", "month", "district_code", "city_code", "district_name", "city_name"] + list(months)
    monthly = monthly[order].sort_values(["district_code", "year", "month"], ignore_index=True)
    monthly[["temp_mean", "temp_max", "temp_min"]] = monthly[["temp_mean", "temp_max", "temp_min"]].round(2)
    monthly["precip_sum"] = monthly["precip_sum"].round(1)

    n_months = (LAST_YEAR - FIRST_YEAR + 1) * 12
    if len(monthly) != len(districts) * n_months or monthly.isna().any().any():
        raise ValueError(f"Expected {len(districts) * n_months} complete rows, got {len(monthly)}")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    output_path = OUTPUT_DIR / f"climate_districts_monthly_{FIRST_YEAR}_{LAST_YEAR}.csv"
    monthly.to_csv(output_path, index=False)
    print(f"Saved {len(monthly)} rows ({len(districts)} districts x {n_months} months) to {output_path} ({output_path.stat().st_size / (1024 * 1024):.1f} MB)")
    monthly.head()
